In [ ]:
!pip install -q transformers accelerate
!pip install -q peft trl bitsandbytes
!pip install -q datasets evaluate bert-score
!pip install -q gradio mlflow wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 103.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 113.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 88.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.

In [ ]:
import datasets
import bitsandbytes as bnb
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

In [ ]:
ds = datasets.load_dataset("sahil2801/CodeAlpaca-20k")
display(ds["train"][0], ds["train"][100], ds["train"][500])

README.md:   0%|          | 0.00/147 [00:00<?, ?B/s]

code_alpaca_20k.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/20022 [00:00<?, ? examples/s]

{'output': 'arr = [2, 4, 6, 8, 10]',
 'instruction': 'Create an array of length 5 which contains all even numbers between 1 and 10.',
 'input': ''}

{'output': 'def find_longest(list):\n    """Return the longest string from a list of strings.""" \n    longest = list[0]\n    for item in list:\n        if len(item) > len(longest):\n            longest = item\n    return longest',
 'instruction': 'Design an algorithm that takes a list of strings and returns the longest string.',
 'input': 'list = ["cat", "dog", "lion", "bird"]'}

{'output': 'DELETE FROM Customer;',
 'instruction': "Write the SQL query to delete all records from the table 'Customer'",
 'input': ''}

In [ ]:
empty_count = len(ds['train'].filter(lambda x: x["input"] == ""))
print(empty_count, len(ds['train']))

Filter:   0%|          | 0/20022 [00:00<?, ? examples/s]

9764 20022


In [ ]:
split_datasets = ds['train'].train_test_split(test_size=0.1, seed=42)
ds_train = split_datasets['train']
ds_test = split_datasets['test']
print(len(ds_train), len(ds_test))

18019 2003


In [ ]:
def sample_to_string(sample: dict) -> str:
  instruction, input, output = sample["instruction"], sample["input"], sample["output"]
  if input == "":
    return f"""### Instruction:
                {instruction}

                ### Response:
                {output}"""
  else:
    return f"""### Instruction:
                {instruction}

                ### Input:
                {input}

                ### Response:
                {output}"""

sample_to_string(ds_train[0])

'### Instruction:\n                Create a Java class which sorts the given array of numbers.\n\n                ### Input:\n                [9, 2, 4, 3, 6, 1]\n\n                ### Response:\n                class ArraySort { \n  \n    void sort(int arr[]) { \n        int n = arr.length; \n  \n        // One by one move boundary of unsorted subarray \n        for (int i = 0; i < n-1; i++) { \n            \n            // Find the minimum element in unsorted array \n            int min_index = i; \n            for (int j = i+1; j < n; j++) \n                if (arr[j] < arr[min_index]) \n                    min_index = j; \n  \n            // Swap the found minimum element with the first element \n            int temp = arr[min_index]; \n            arr[min_index] = arr[i]; \n            arr[i] = temp; \n        } \n    } \n  \n    // Prints the array \n    void printArray(int arr[]) { \n        int n = arr.length; \n        for (int i=0; i<n; ++i) \n            System.out.print(arr[

In [ ]:
bnb_config  = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

In [12]:
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-3B", padding_side="right")
model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-3B",
                                              quantization_config=bnb_config ,
                                              device_map="auto"
                                             )

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

In [13]:
model.get_memory_footprint()

2197641728

In [28]:
test_prompt = sample_to_string(ds_train[0])
print(test_prompt)

test_prompt = test_prompt.split("### Response")[0]
test_prompt += "\n### Response:\n"
print(test_prompt)

### Instruction:
                Create a Java class which sorts the given array of numbers.

                ### Input:
                [9, 2, 4, 3, 6, 1]

                ### Response:
                class ArraySort { 
  
    void sort(int arr[]) { 
        int n = arr.length; 
  
        // One by one move boundary of unsorted subarray 
        for (int i = 0; i < n-1; i++) { 
            
            // Find the minimum element in unsorted array 
            int min_index = i; 
            for (int j = i+1; j < n; j++) 
                if (arr[j] < arr[min_index]) 
                    min_index = j; 
  
            // Swap the found minimum element with the first element 
            int temp = arr[min_index]; 
            arr[min_index] = arr[i]; 
            arr[i] = temp; 
        } 
    } 
  
    // Prints the array 
    void printArray(int arr[]) { 
        int n = arr.length; 
        for (int i=0; i<n; ++i) 
            System.out.print(arr[i] + " "); 
        System.out.pr

In [24]:
input_tokenized = tokenizer(test_prompt,
    return_tensors="pt",
    truncation=True,
    max_length=512
)
input_tokenized = input_tokenized.to("cuda")
print(input_tokenized["input_ids"].device)

cuda:0


In [25]:
result = model.generate(
    **input_tokenized,
    max_new_tokens=200,
    do_sample=False,
    pad_token_id=tokenizer.eos_token_id,
    repetition_penalty=1.1
)

In [26]:
generated_only = result[0][input_tokenized["input_ids"].shape[1]:]
output = tokenizer.decode(generated_only, skip_special_tokens=True)
print(output)

[1, 2, 3, 4, 6, 9]

